In [2]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import xgboost as xgb
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import joblib
import os

In [3]:
CSV_PATH       = r"..\PREPROCESSING\processed_cicids2017.csv"
BOTTLENECK_DIM = 16
EPOCHS         = 30
BATCH_SIZE     = 2048
LR             = 1e-3
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SAVE_DIR       = "../MODELS"
os.makedirs(SAVE_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────
print("Loading data...")
df = pd.read_csv(CSV_PATH)

Loading data...


In [4]:
label_col   = "Label"
attack_col  = "Attack_Type"   # change to your actual column name if different

feature_cols = [c for c in df.columns if c not in [label_col, attack_col]]

X = df[feature_cols].values.astype(np.float32)
y_binary = df[label_col].values  # 0 or 1

# ─────────────────────────────────────────────
# 2. SCALE
# ─────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
joblib.dump(scaler, f"{SAVE_DIR}/scaler.pkl")

['../MODELS/scaler.pkl']

In [5]:
X_benign = X_scaled[y_binary == 0]
print(f"Benign samples for AE training: {len(X_benign):,}")

X_ae_train, X_ae_val = train_test_split(X_benign, test_size=0.1, random_state=42)

ae_train_loader = DataLoader(
    TensorDataset(torch.tensor(X_ae_train)),
    batch_size=BATCH_SIZE, shuffle=True
)
ae_val_loader = DataLoader(
    TensorDataset(torch.tensor(X_ae_val)),
    batch_size=BATCH_SIZE
)

Benign samples for AE training: 2,271,320


In [14]:
input_dim = X_scaled.shape[1]

class Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, input_dim)
        )

    def forward(self, x):        # ← must be INSIDE the same cell as the class
        z = self.encoder(x)
        return self.decoder(z), z

ae = Autoencoder(input_dim, BOTTLENECK_DIM).to(DEVICE)
optimizer = torch.optim.Adam(ae.parameters(), lr=LR)
criterion = nn.MSELoss()

In [15]:

print("\nTraining Autoencoder on benign data...")
ae = Autoencoder(input_dim, BOTTLENECK_DIM).to(DEVICE)
optimizer = torch.optim.Adam(ae.parameters(), lr=LR)
criterion = nn.MSELoss()

for epoch in range(EPOCHS):
    ae.train()
    train_loss = 0
    for (batch,) in ae_train_loader:
        batch = batch.to(DEVICE)
        recon, _ = ae(batch)
        loss = criterion(recon, batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(batch)

    ae.eval()
    val_loss = 0
    with torch.no_grad():
        for (batch,) in ae_val_loader:
            batch = batch.to(DEVICE)
            recon, _ = ae(batch)
            val_loss += criterion(recon, batch).item() * len(batch)

    print(f"Epoch {epoch+1:02d}/{EPOCHS} | "
          f"Train Loss: {train_loss/len(X_ae_train):.6f} | "
          f"Val Loss:   {val_loss/len(X_ae_val):.6f}")

torch.save(ae.state_dict(), f"{SAVE_DIR}/autoencoder.pt")
print("Autoencoder saved.")


Training Autoencoder on benign data...
Epoch 01/30 | Train Loss: 0.194836 | Val Loss:   0.083243
Epoch 02/30 | Train Loss: 0.091658 | Val Loss:   0.075845
Epoch 03/30 | Train Loss: 0.064404 | Val Loss:   0.052356
Epoch 04/30 | Train Loss: 0.062440 | Val Loss:   0.079649
Epoch 05/30 | Train Loss: 0.057389 | Val Loss:   0.031470
Epoch 06/30 | Train Loss: 0.049761 | Val Loss:   0.082891
Epoch 07/30 | Train Loss: 0.069897 | Val Loss:   0.024093
Epoch 08/30 | Train Loss: 0.049775 | Val Loss:   0.024967
Epoch 09/30 | Train Loss: 0.036118 | Val Loss:   0.029335
Epoch 10/30 | Train Loss: 0.053090 | Val Loss:   0.075253
Epoch 11/30 | Train Loss: 0.068342 | Val Loss:   0.057220
Epoch 12/30 | Train Loss: 0.045181 | Val Loss:   0.044971
Epoch 13/30 | Train Loss: 0.041434 | Val Loss:   0.024308
Epoch 14/30 | Train Loss: 0.043692 | Val Loss:   0.022524
Epoch 15/30 | Train Loss: 0.036942 | Val Loss:   0.077003
Epoch 16/30 | Train Loss: 0.025227 | Val Loss:   0.044755
Epoch 17/30 | Train Loss: 0.0327

In [16]:
print("\nExtracting AE features from all data...")
ae.eval()

def extract_features(X_np, batch_size=4096):
    all_errors, all_bottlenecks = [], []
    tensor = torch.tensor(X_np, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size)
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            recon, z = ae(batch)
            per_feature_error = (batch - recon) ** 2   # shape: (N, input_dim)
            all_errors.append(per_feature_error.cpu().numpy())
            all_bottlenecks.append(z.cpu().numpy())
    return np.concatenate(all_errors), np.concatenate(all_bottlenecks)

errors, bottlenecks = extract_features(X_scaled)
X_xgb = np.concatenate([errors, bottlenecks], axis=1)
print(f"XGBoost feature matrix shape: {X_xgb.shape}")


Extracting AE features from all data...
XGBoost feature matrix shape: (2827876, 63)


In [17]:
print("\nTraining Model 1: Binary XGBoost...")
X_train1, X_test1, y_train1, y_test1 = train_test_split(
    X_xgb, y_binary, test_size=0.2, random_state=42, stratify=y_binary
)

# Handle class imbalance: 2271320 benign vs 556556 anomaly
scale_pos_weight = (y_binary == 0).sum() / (y_binary == 1).sum()
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

model1 = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    use_label_encoder=False,
    eval_metric="logloss",
    tree_method="hist",          # fast, works on CPU and GPU
    device="cuda" if torch.cuda.is_available() else "cpu",
    random_state=42
)



Training Model 1: Binary XGBoost...
scale_pos_weight: 4.08


In [18]:
model1.fit(
    X_train1, y_train1,
    eval_set=[(X_test1, y_test1)],
    verbose=50
)

print("\nModel 1 — Classification Report:")
y_pred1 = model1.predict(X_test1)
print(classification_report(y_test1, y_pred1, target_names=["Benign", "Anomaly"]))
print("Confusion Matrix:\n", confusion_matrix(y_test1, y_pred1))

model1.save_model(f"{SAVE_DIR}/model1_binary.ubj")
print("Model 1 saved.")

c:\Users\sudee\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:200: UserWarning: [22:02:54] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[0]	validation_0-logloss:0.61173
[50]	validation_0-logloss:0.05490
[100]	validation_0-logloss:0.03266
[150]	validation_0-logloss:0.02485
[200]	validation_0-logloss:0.02006
[250]	validation_0-logloss:0.01722
[299]	validation_0-logloss:0.01541

Model 1 — Classification Report:
              precision    recall  f1-score   support

      Benign       1.00      0.99      1.00    454265
     Anomaly       0.97      1.00      0.99    111311

    accuracy                           0.99    565576
   macro avg       0.99      1.00      0.99    565576
weighted avg       0.99      0.99      0.99    565576

Confusion Matrix:
 [[451327   2938]
 [   248 111063]]
Model 1 saved.


In [20]:
test=pd.read_csv("..\\DATASET\\stage2_test_original.csv")

In [21]:
test.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Backward Packets,Total Length of Fwd Packets,Total Length of Bwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,...,min_seg_size_forward,Active Mean,Active Std,Active Max,Active Min,Idle Mean,Idle Std,Idle Max,Idle Min,Label
0,80.0,1000000.0,3.0,4.0,26.0,11607.0,20.0,0.0,8.666667,10.263203,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
1,80.0,1000000.0,5.0,7.0,390.0,11595.0,378.0,0.0,78.000000,167.731929,...,20.0,20010.0,0.0,20010.0,20010.0,1000000.0,0.0,1000000.0,1000000.0,1
2,49159.0,108.0,1.0,1.0,2.0,6.0,2.0,2.0,2.000000,0.000000,...,24.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,2
3,80.0,14548.0,2.0,2.0,12.0,0.0,6.0,6.0,6.000000,0.000000,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1
4,80.0,1000000.0,3.0,5.0,26.0,11601.0,20.0,0.0,8.666667,10.263203,...,20.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1


In [ ]:
# Use your actual test split from training
real_sample_idx = np.random.choice(len(X_test1), size=200, replace=False)
real_sample_X   = X_test1[real_sample_idx]
real_sample_y   = y_test1[real_sample_idx]

real_preds = model1.predict(real_sample_X)
real_probs = model1.predict_proba(real_sample_X)[:, 1]

print(classification_report(real_sample_y, real_preds, target_names=["Benign", "Anomaly"]))

print(f"\n{'Sample':<10} {'True':<10} {'Predicted':<12} {'Anomaly Prob'}")
print("-" * 45)
for i in range(len(real_sample_y)):
    true_label = "Benign"  if real_sample_y[i] == 0 else "Anomaly"
    pred_label = "Benign"  if real_preds[i] == 0    else "Anomaly"
    flag = "✓" if real_sample_y[i] == real_preds[i] else "✗"
    print(f"{i:<10} {true_label:<10} {pred_label:<12} {real_probs[i]:.4f}  {flag}")

In [1]:
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from sklearn.metrics import classification_report, confusion_matrix

# ─────────────────────────────────────────────
# CONFIG — adjust paths if needed
# ─────────────────────────────────────────────
CSV_PATH      = r"..\PREPROCESSING\processed_cicids2017.csv"
MODELS_DIR    = r"..\MODELS"
BOTTLENECK_DIM = 16
DEVICE        = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ─────────────────────────────────────────────
# 1. LOAD SCALER + DATA
# ─────────────────────────────────────────────
print("Loading scaler and data...")
scaler = joblib.load(f"{MODELS_DIR}/scaler.pkl")

df = pd.read_csv(CSV_PATH)
label_col    = "Label"
attack_col   = "Attack_Type"
feature_cols = [c for c in df.columns if c not in [label_col, attack_col]]

X        = df[feature_cols].values.astype(np.float32)
y_binary = df[label_col].values

X_scaled = scaler.transform(X)   # use transform only, NOT fit_transform
print(f"Data shape: {X_scaled.shape}")

# ─────────────────────────────────────────────
# 2. LOAD AUTOENCODER
# ─────────────────────────────────────────────
class Autoencoder(nn.Module):
    def __init__(self, input_dim, bottleneck_dim=16):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(),
            nn.Linear(128, 64),        nn.ReLU(),
            nn.Linear(64, bottleneck_dim)
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 64), nn.ReLU(),
            nn.Linear(64, 128),            nn.ReLU(),
            nn.Linear(128, input_dim)
        )
    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z

input_dim   = X_scaled.shape[1]
ae          = Autoencoder(input_dim, BOTTLENECK_DIM).to(DEVICE)
ae.load_state_dict(torch.load(f"{MODELS_DIR}/autoencoder.pt", map_location=DEVICE))
ae.eval()
print("Autoencoder loaded.")

# ─────────────────────────────────────────────
# 3. EXTRACT AE FEATURES
# ─────────────────────────────────────────────
def extract_features(X_np, batch_size=4096):
    all_errors, all_bottlenecks = [], []
    tensor = torch.tensor(X_np, dtype=torch.float32)
    loader = DataLoader(TensorDataset(tensor), batch_size=batch_size)
    with torch.no_grad():
        for (batch,) in loader:
            batch = batch.to(DEVICE)
            recon, z = ae(batch)
            all_errors.append((batch - recon).pow(2).cpu().numpy())
            all_bottlenecks.append(z.cpu().numpy())
    return np.concatenate(all_errors), np.concatenate(all_bottlenecks)

print("Extracting AE features from real data (this may take a minute)...")
errors, bottlenecks = extract_features(X_scaled)
X_xgb = np.concatenate([errors, bottlenecks], axis=1)
print(f"AE feature matrix shape: {X_xgb.shape}")

# ─────────────────────────────────────────────
# 4. LOAD MODEL 1
# ─────────────────────────────────────────────
model1 = xgb.XGBClassifier()
model1.load_model(f"{MODELS_DIR}/model1_binary.ubj")
print("Model 1 loaded.")

# ─────────────────────────────────────────────
# 5. SAMPLE REAL DATA FOR TESTING
#    500 benign + 500 anomaly for balanced view
# ─────────────────────────────────────────────
np.random.seed(42)
benign_idx  = np.where(y_binary == 0)[0]
anomaly_idx = np.where(y_binary == 1)[0]

sample_benign  = np.random.choice(benign_idx,  size=500, replace=False)
sample_anomaly = np.random.choice(anomaly_idx, size=500, replace=False)
sample_idx     = np.concatenate([sample_benign, sample_anomaly])

X_test_real = X_xgb[sample_idx]
y_test_real = y_binary[sample_idx]

# ─────────────────────────────────────────────
# 6. PREDICT + EVALUATE
# ─────────────────────────────────────────────
print("\nRunning predictions on 1000 real samples (500 benign + 500 anomaly)...")
y_pred = model1.predict(X_test_real)
y_prob = model1.predict_proba(X_test_real)[:, 1]

print("\n=== Model 1 — Real Data Test Results ===")
print(classification_report(y_test_real, y_pred, target_names=["Benign", "Anomaly"]))
print("Confusion Matrix:")
print(confusion_matrix(y_test_real, y_pred))

print(f"\n{'Sample':<10} {'True':<10} {'Predicted':<12} {'Anomaly Prob'}")
print("-" * 48)
for i in range(len(y_test_real)):
    true_label = "Benign"  if y_test_real[i] == 0 else "Anomaly"
    pred_label = "Benign"  if y_pred[i] == 0       else "Anomaly"
    flag       = "✓" if y_test_real[i] == y_pred[i] else "✗"
    print(f"{i:<10} {true_label:<10} {pred_label:<12} {y_prob[i]:.4f}  {flag}")

Loading scaler and data...
Data shape: (2827876, 47)
Autoencoder loaded.
Extracting AE features from real data (this may take a minute)...
AE feature matrix shape: (2827876, 63)
Model 1 loaded.

Running predictions on 1000 real samples (500 benign + 500 anomaly)...

=== Model 1 — Real Data Test Results ===
              precision    recall  f1-score   support

      Benign       1.00      0.99      1.00       500
     Anomaly       0.99      1.00      1.00       500

    accuracy                           1.00      1000
   macro avg       1.00      1.00      1.00      1000
weighted avg       1.00      1.00      1.00      1000

Confusion Matrix:
[[496   4]
 [  0 500]]

Sample     True       Predicted    Anomaly Prob
------------------------------------------------
0          Benign     Benign       0.0000  ✓
1          Benign     Benign       0.0001  ✓
2          Benign     Benign       0.0315  ✓
3          Benign     Benign       0.0000  ✓
4          Benign     Benign       0.0007  ✓
5

In [1]:
import torch
ckpt = torch.load(r"C:\Users\sudee\OneDrive\Documents\GitHub\MINOR_2-NETWORK-ANAMOLY-DETECTION\MODELS\autoencoder.pt")
print(type(ckpt))
print(ckpt.keys() if isinstance(ckpt, dict) else "Not a dict — direct state_dict")

<class 'collections.OrderedDict'>
odict_keys(['encoder.0.weight', 'encoder.0.bias', 'encoder.2.weight', 'encoder.2.bias', 'encoder.4.weight', 'encoder.4.bias', 'decoder.0.weight', 'decoder.0.bias', 'decoder.2.weight', 'decoder.2.bias', 'decoder.4.weight', 'decoder.4.bias'])


In [4]:
import joblib
scaler = joblib.load(r"..\MODELS\scaler.pkl")
print("N features:", scaler.n_features_in_)

# Check if feature names were saved
if hasattr(scaler, 'feature_names_in_'):
    print("Feature names:", list(scaler.feature_names_in_))
else:
    print("No feature names saved in scaler")

N features: 47
No feature names saved in scaler


In [8]:
import joblib
import pandas as pd

feat_cols = joblib.load(r"..\MODELS\model2_feature_cols.pkl")
print("Model 2 feat_cols count:", len(feat_cols))

scaler = joblib.load(r"..\MODELS\scaler.pkl")
print("Scaler expects:", scaler.n_features_in_)

df = pd.read_csv(r"..\PREPROCESSING\processed_cicids2017.csv")
df.columns = df.columns.str.strip()
feature_cols_training = [c for c in df.columns if c not in ["Label", "Attack_Type"]]
print("Training feature count:", len(feature_cols_training))
print("Training features:", feature_cols_training)

Model 2 feat_cols count: 78
Scaler expects: 47
Training feature count: 47
Training features: ['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Min', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Bwd Packets/s', 'Min Packet Length', 'FIN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'Down/Up Ratio', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Std', 'Active Max', '